In [ ]:
import time

from langchain_classic.embeddings import CacheBackedEmbeddings
from langchain_classic.storage import LocalFileStore
from langchain_openai import OpenAIEmbeddings


embedding_model = OpenAIEmbeddings(
    model="text-embedding-3-small"
)

library_rule_chunks = [
    (
        "[2026학년도 중앙도서관 대출 및 열람 규정]\n"
        "제1조(대출 자격) 본교 재학생 및 휴학생은 "
        "학생증을 제시한 후 도서를 대출할 수 있습니다."
    ),
    (
        "제2조(대출 권수 및 기간) 학부생은 1인당 최대 5권까지 "
        "대출할 수 있으며 대출 기간은 14일입니다.\n"
        "단, 학술 연구용 특수 도서는 7일간 대출 가능하며 "
        "연장은 불가능합니다."
    ),
    (
        "제3조(연체 및 제재) 반납 기한을 초과할 경우 "
        "연체 일수만큼 대출이 정지됩니다."
    ),
]

store = LocalFileStore("./.cache/")

cache_embedder = CacheBackedEmbeddings.from_bytes_store(
    underlying_embeddings=embedding_model,
    document_embedding_cache=store,
    namespace="library_rules",
)

In [ ]:
# 최초 실행 - 임베딩 생성 -> 캐시 기록 
# 성능에 대한 측정은 time 모듈을 활용하여 수행합니다.
start_time = time.time()
vector_first = cache_embedder.embed_documents(library_rule_chunks)
end_time = time.time() - start_time #현재 소요시간
print(f"1번째 실행 소요시간: {end_time:.4f}초")

# 두 번째 실행 - 캐시에서 임베딩 조회
# 성능에 대한 측정은 time 모듈을 활용하여 수행합니다.
# 최초 모델보다 불러오는 시간이 짧아진다. <- 이 맥락이 중요!!!!(file쪽에서 불러오므로)
start_time = time.time()
vector_second = cache_embedder.embed_documents(library_rule_chunks)
end_time = time.time() - start_time #현재 소요시간
print(f"2번째 실행 소요시간: {end_time:.4f}초")


print(f"동일한 벡터값: {vector_first==vector_second}")
